# Step 4-1. Agent: LLM이 스스로 MCP 도구를 고른다
강의에서 본 **ReAct(리액트)** 를 직접 만듭니다. 프레임워크 없이 `for`문 하나입니다.

## Agent의 반복
1. LLM에게 질문과 **도구 목록**을 함께 준다
2. LLM이 "이 도구를 이 인자로 불러 달라"고 답하면, 우리가 MCP 서버에 대신 호출한다
3. 결과를 LLM에게 돌려준다
4. LLM이 도구를 더 부르지 않고 답을 쓰면 끝

모델은 내 맥북의 로컬 LLM입니다. 요금이 없습니다.

In [ ]:
# 필요한 패키지를 설치합니다. 처음에는 1분쯤 걸립니다.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "mcp>=2.3", "chromadb", "ollama"], check=True)
print("✅ 패키지 준비 완료. 다음 셀로 넘어가세요.")

## 1. 설정

In [ ]:
import json

try:
    with open("../config.json") as f:
        _config = json.load(f)
    print("✅ config.json 로드 완료")
except FileNotFoundError:
    raise SystemExit("❌ config.json이 없습니다. Step 0 노트북을 먼저 실행해 주세요.")

EMBED_MODEL = _config["EMBED_MODEL"]
LLM_MODEL = _config["LLM_MODEL"]
AGENT_MODEL = _config["AGENT_MODEL"]
print(f"   Embedding 모델: {EMBED_MODEL}")
print(f"   LLM 모델:       {LLM_MODEL}")
print(f"   Agent 모델:     {AGENT_MODEL}")

## 2. MCP 도구 목록을 LLM이 이해하는 형식으로
MCP가 알려 준 "이름, 설명, 인자"를 그대로 LLM에게 넘깁니다.

In [ ]:
import os
import sys
from mcp import Client, StdioServerParameters

SERVER_PATH = os.path.abspath("../../memory_mcp.py")
server = StdioServerParameters(command=sys.executable, args=[SERVER_PATH], env=os.environ.copy())

async with Client(server) as mcp_client:
    mcp_tools = (await mcp_client.list_tools()).tools

llm_tools = [
    {"type": "function", "function": {"name": t.name, "description": t.description, "parameters": t.input_schema}}
    for t in mcp_tools
]
print("✅ LLM에게 줄 도구:", ", ".join(t.name for t in mcp_tools))

## 3. Agent 만들기
아래 `for`문이 Agent의 전부입니다. 🧠는 LLM의 판단, 🔧는 도구 호출, 📥는 도구가 돌려준 결과입니다.

In [ ]:
import json
import ollama

SYSTEM = """너는 사용자의 개인 비서다.
- 기억해 달라는 말에는 remember 도구를 쓴다.
- 예전 일이나 저장해 둔 내용을 묻는 말에는 recall 도구로 먼저 찾는다.
- 도구가 돌려준 내용만 근거로, 한국어로 짧게 답한다. 찾지 못했으면 모른다고 답한다."""


async def ask_agent(question, max_steps=5):
    print("=" * 60)
    print(f"👤 사용자: {question}")
    print("=" * 60)
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]

    async with Client(server) as mcp_client:
        for step in range(max_steps):
            reply = ollama.chat(model=AGENT_MODEL, messages=messages, tools=llm_tools, options={"temperature": 0.2})["message"]
            messages.append(reply)

            if not reply.tool_calls:                      # 도구를 더 부르지 않으면 최종 답
                print(f"🤖 답: {reply.content.strip()}\n")
                return

            for tc in reply.tool_calls:                   # LLM이 고른 도구를 대신 실행
                name, args = tc.function.name, dict(tc.function.arguments)
                print(f"🧠 {step + 1}번째 판단 → 🔧 {name}({json.dumps(args, ensure_ascii=False)})")
                try:
                    r = await mcp_client.call_tool(name, args)
                    result = r.content[0].text if r.content else "(결과 없음)"
                except Exception as e:
                    result = f"도구 오류: {e}"
                print(f"📥 결과: {result[:200]}")
                messages.append({"role": "tool", "tool_name": name, "content": result})

    print("⚠️ 정해 둔 횟수 안에 끝나지 않았습니다.\n")

## 4. 실행
### 테스트 1: 기억시키기
`remember` 도구를 고르는지 보세요.

In [ ]:
# TODO: 기억할 내용을 바꿔 보세요!
await ask_agent("이거 기억해 줘: 목요일 오후 3시에 영상의학과 판독 회의")

### 테스트 2: 꺼내기
질문에는 "회의"라는 말만 있습니다. `recall`로 찾아서 답하는지 보세요.

In [ ]:
# TODO: 질문을 바꿔 보세요!
await ask_agent("이번 주에 회의가 언제 있었지?")

### 테스트 3: 도구가 필요 없는 질문
도구 없이 답할 수 있는 질문입니다. 작은 모델은 필요 없는 도구를 부르기도 합니다. 어떻게 하는지 보세요.

In [ ]:
await ask_agent("1 더하기 1은?")

## 정리
- Agent는 "LLM + 도구 목록 + 반복"입니다. 80줄짜리 MCP 서버와 `for`문 하나로 만들었습니다.
- 작은 로컬 모델은 가끔 도구를 안 부르거나 인자를 틀립니다. Step 0에서 `AGENT_MODEL`을 `gemma4:e4b`로 바꾸거나, 같은 MCP 서버를 Claude에 붙이면 훨씬 안정적입니다.
- **같은 `memory_mcp.py`를 Claude에 붙이는 방법**은 README의 "Claude에 연결"을 보세요. 이 노트북에서 저장한 기억을 Claude가 그대로 꺼냅니다.